In [8]:
import json
import gzip
import os
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from scipy.special import expit as sigmoid
from sklearn.metrics import f1_score
from skmultilearn.model_selection import iterative_train_test_split
import optuna

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)


In [2]:
from datasets import load_from_disk

dataset = load_from_disk("../binary_dataset")

train_dataset = dataset["train"]
dev_dataset = dataset["validation"]
test_dataset = dataset["test"]

In [3]:
train_dataset

Dataset({
    features: ['u', 'id', 'ts', 'text', 'Turku_NLP', 'Turku_NLP_sub', 'Tags', 'web-register', 'Binary', 'source'],
    num_rows: 3577
})

In [7]:
print(train_dataset.features)
print(train_dataset[0]["Binary"])
print(type(train_dataset[0]["Binary"]))

{'u': Value('string'), 'id': Value('string'), 'ts': Value('string'), 'text': Value('string'), 'Turku_NLP': Value('string'), 'Turku_NLP_sub': Value('string'), 'Tags': Value('string'), 'web-register': Value('string'), 'Binary': ClassLabel(names=['0', '1']), 'source': Value('string')}
1
<class 'int'>


In [10]:
from collections import Counter

print(Counter(train_dataset["Binary"]))

Counter({1: 3163, 0: 414})


In [ ]:
NUM_LABELS = 2

In [ ]:
MODEL_NAME = "BAAI/bge-m3-retromae"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=1024,
    )

train_dataset = train_dataset.map(tokenize, batched=True)
dev_dataset = dev_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

In [ ]:
for ds in (train_dataset, dev_dataset, test_dataset):
    ds.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"],
    )

In [ ]:
MODEL_NAME = "BAAI/bge-m3-retromae"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=1024,
    )

train_dataset = train_dataset.map(tokenize, batched=True)
dev_dataset = dev_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

In [ ]:
for ds in (train_dataset, dev_dataset, test_dataset):
    ds.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"],
    )

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

For single-label binary (or multiclass) classification:

Micro F1 is mathematically identical to accuracy.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }

In [ ]:
# ------------------------------------------------
# Optuna objective
# ------------------------------------------------

def objective(trial):

    learning_rate = trial.suggest_float("learning_rate", 5e-6, 3e-5, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.0, 0.15)

    per_device_batch = trial.suggest_categorical("batch_size", [4, 8])
    grad_accum = trial.suggest_categorical("grad_accum", [4, 8])

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        problem_type="multi_label_classification",
    )

    args = TrainingArguments(
        output_dir=f"./optuna_xlmr/trial_{trial.number}",
        overwrite_output_dir=True,

        num_train_epochs=10,
        per_device_train_batch_size=per_device_batch,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=grad_accum,

        learning_rate=learning_rate,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,

        eval_strategy="epoch",
        logging_strategy="epoch",

        save_strategy="no",
        report_to="none",

        seed=42,
        bf16=True,
        metric_for_best_model="f1",
        greater_is_better=True,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    trainer.train()
    metrics = trainer.evaluate()

    return metrics["eval_f1_micro"]